# 8장 · temperature — 창의성 다이얼의 정체 🌡️

웹 챕터 **"temperature"** 와 짝을 이루는 Python 실습입니다.
웹 다이얼에서 잡은 감(T = 엔트로피 조절기)을, 여기서는 **H(T) 곡선과
T별 100회 생성 실험**으로 굳힙니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch8')

## 1. softmax(z/T): 다이얼 달린 소프트맥스

로짓은 고정하고 T 만 바꿔 분포가 어떻게 변하는지 나란히 그려 봅니다.
예문: "오늘 날씨가 참 ___" 의 다음 토큰 5개.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

def softmax_T(z, T=1.0):
    """다이얼 달린 소프트맥스 — 로짓을 T 로 나눈 뒤 softmax."""
    z = np.asarray(z, dtype=float) / T
    e = np.exp(z - z.max())
    return e / e.sum()

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(np.sum(p * np.log2(1 / p)))

tokens = ['좋다', '맑다', '춥다', '이상하다', '감자같다']
logits = np.array([3.0, 2.2, 1.5, 0.3, -1.5])   # 모델의 점수 — 이건 고정!

Ts = [0.2, 0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(1, len(Ts), figsize=(14, 3.2), sharey=True)
for ax, T in zip(axes, Ts):
    q = softmax_T(logits, T)
    ax.bar(tokens, q, color='#6366f1', alpha=0.85)
    ax.set_title(f'T={T}\nH={entropy(q):.2f}비트', fontsize=10)
    ax.tick_params(axis='x', labelsize=7, rotation=45)
axes[0].set_ylabel('확률')
fig.suptitle('같은 로짓, 다른 온도: T 가 클수록 균등(높은 H), 작을수록 몰빵(낮은 H)')
plt.tight_layout()
plt.show()

## 2. H(T) 곡선: 다이얼과 엔트로피의 관계를 한 장에

T 를 0.05부터 20까지 돌리며 엔트로피를 기록합니다.
**단조 증가**(다이얼을 올리면 H 도 반드시 올라감)와 **양 끝의 극한**을 확인하세요.

In [ ]:
Ts_fine = np.linspace(0.05, 20, 400)
Hs = np.array([entropy(softmax_T(logits, T)) for T in Ts_fine])
H_max = np.log2(len(tokens))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(Ts_fine, Hs, color='#6366f1', lw=2.5)
ax.axhline(H_max, color='#ef4444', lw=1.5, linestyle='--', label=f'최대 log2(5) = {H_max:.2f}비트 (균등)')
ax.axhline(0, color='#94a3b8', lw=1, linestyle=':')
ax.set_xlabel('temperature T')
ax.set_ylabel('엔트로피 H (비트)')
ax.set_title('T = 엔트로피 다이얼: 0(몰빵)에서 log2(n)(균등)까지 단조 증가')
ax.grid(True, color='#eef2f7')
ax.legend()
plt.show()

assert np.all(np.diff(Hs) >= -1e-9), '다이얼을 올렸는데 H 가 내려갔다?!'
assert Hs[0] < 0.1 and Hs[-1] > H_max - 0.1
print(f'T=0.05 → H={Hs[0]:.4f}비트 (거의 0) · T=20 → H={Hs[-1]:.4f}비트 (거의 최대 {H_max:.4f})')

## 3. 생성 실험: 같은 모델, 다른 다이얼로 100번씩

웹의 🎲 버튼을 대량으로: T 별로 100번 샘플링해서 뭐가 몇 번 나왔는지 셉니다.
"창의성 = 꼬리 토큰의 기회"를 숫자로 확인하세요.

In [ ]:
rng = np.random.default_rng(42)

print('T    | ' + ' | '.join(f'{t:>6s}' for t in tokens) + ' | 등장한 종류')
print('-' * 66)
for T in [0.2, 1.0, 2.0, 5.0]:
    q = softmax_T(logits, T)
    draws = rng.choice(len(tokens), size=100, p=q)
    counts = np.bincount(draws, minlength=len(tokens))
    kinds = int((counts > 0).sum())
    print(f'{T:<4} | ' + ' | '.join(f'{c:6d}' for c in counts) + f' | {kinds}종류')

print('\n🧊 T=0.2: 사실상 "좋다" 복사기 · 🔥 T=5: "감자같다"까지 골고루. 다양성의 대가는 헛소리 위험.')

## 4. top-p 가위: 꼬리를 자르는 법

temperature 가 분포의 **모양**을 바꾼다면, top-p 는 **꼬리를 자르는 가위**입니다.
누적 확률이 p 를 넘을 때까지의 상위 토큰만 남기고 재정규화하는 방법을 직접 구현해 봅니다.

In [ ]:
def top_p_filter(q, p_keep=0.9):
    """누적 확률 p_keep 까지의 상위 토큰만 남기고 재정규화 (nucleus 샘플링의 심장)."""
    order = np.argsort(q)[::-1]                # 확률 내림차순 인덱스
    cum = np.cumsum(q[order])
    cut = int(np.searchsorted(cum, p_keep)) + 1  # 누적이 p_keep 를 넘는 첫 지점까지 포함
    keep = order[:cut]
    out = np.zeros_like(q)
    out[keep] = q[keep] / q[keep].sum()
    return out

for T in [1.0, 3.0]:
    q = softmax_T(logits, T)
    filtered = top_p_filter(q, 0.9)
    survivors = [tokens[i] for i in range(len(tokens)) if filtered[i] > 0]
    print(f'T={T}: 원본 q = {np.round(q, 3)}')
    print(f'      top-p 0.9 생존자: {survivors} ({len(survivors)}개)\n')

print('확신 상태(T=1)에서는 후보를 좁게, 헷갈리는 상태(T=3)에서는 넓게.')
print('후보 수가 상황 따라 변하는 "동적 가위"입니다. 실전은 T 로 모양 잡고 top-p 로 극단만 자르죠.')

## 5. 직접 풀어보기 ✏️

T 를 아주 크게(예: 1,000,000) 올리면 이 5개 토큰 분포의 엔트로피는 어디에 도달할까요?
(웹 챕터의 보통 문제와 같은 계산, 비트 단위)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: T→∞ 는 균등 분포 — 균등 n갈래의 H = log₂n

answer = entropy(softmax_T(logits, T=1_000_000))
print('내 답:', answer)
assert abs(answer - np.log2(5)) < 1e-3, '다시 생각해 보세요 — 점수 차이가 소멸하면 어떤 분포가 되나요?'
print(f'✅ 잘했어요! log₂5 ≈ {np.log2(5):.4f}비트. 다이얼의 오른쪽 끝은 균등의 엔트로피입니다.')

---
**오늘의 한 줄**: temperature = softmax(z/T) 의 T. 분포의 엔트로피를 0(몰빵)에서
log₂n(균등)까지 돌리는 다이얼. 창의성의 실체는 꼬리 토큰의 기회다.

다음 챕터는 심화 트랙입니다. **문맥은 왜 예측을 쉽게 만드는가?** 조건부 엔트로피 H(X|Y) 와
상호정보량 I(X;Y) 로 답합니다. 🙂